In [50]:
import pandas as pd
from tqdm.auto import tqdm
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgba

tqdm.pandas()

Load the Data

In [67]:
spotify_tracks = pd.read_csv('spotify_tracks_pop.csv')
display(pd.read_csv('spotify_tracks_pop.csv').head(5))
spotify_tracks['track_album_release_date'] = pd.to_datetime(spotify_tracks['track_album_release_date'])
spotify_tracks.shape


,id,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,track_album_release_date,...,key,duration_min,acousticness,first_listed_artist,track_album_name,playlist_name,playlist_genre,playlist_subgenre,track_name,track_popularity
0,2plbrEY59IikOBgBGLjaoe,0.592,157.969,0.521,-7.777,0.122,0.535,3/4,0.0304,2024-08-16,...,6.0,4.194467,0.3080,Lady Gaga,Die With A Smile,Today's Top Hits,pop,mainstream,Die With A Smile,100
1,2plbrEY59IikOBgBGLjaoe,0.592,157.969,0.521,-7.777,0.122,0.535,3/4,0.0304,2024-08-16,...,6.0,4.194467,0.3080,Lady Gaga,Die With A Smile,Global Top 50,pop,global,Die With A Smile,100
2,2plbrEY59IikOBgBGLjaoe,0.592,157.969,0.521,-7.777,0.122,0.535,3/4,0.0304,2024-08-16,...,6.0,4.194467,0.3080,Lady Gaga,Die With A Smile,Top Gaming Tracks,gaming,modern,Die With A Smile,100
3,5vNRhkKd0yEAg8suGBpjeY,0.783,149.027,0.777,-4.477,0.355,0.939,4/4,0.2600,2024-10-18,...,0.0,2.831950,0.0283,ROSÉ,APT.,Today's Top Hits,pop,mainstream,APT.,98
4,5vNRhkKd0yEAg8suGBpjeY,0.783,149.027,0.777,-4.477,0.355,0.939,4/4,0.2600,2024-10-18,...,0.0,2.831950,0.0283,ROSÉ,APT.,Global Top 50,pop,global,APT.,98


(4684, 22)

Question: Is song popularity associated with musical mode (Major, Minor) within the five most represented genres in our dataset?

In [88]:
top_5_genres = spotify_tracks["playlist_genre"].value_counts().head(5).index.tolist()

print(top_5_genres)

['electronic', 'pop', 'latin', 'hip-hop', 'ambient']


Creating a new data set of top 5 genres and removing duplicates in our newly constructed dataset to ensure unadulterated statistical evaluation

In [93]:
top_5_data = spotify_tracks[spotify_tracks["playlist_genre"].isin(top_5_genres)].copy()
top_5_data = top_5_data.drop_duplicates(subset=["id"])

In [96]:
major_minor_data = top_5_data[top_5_data["mode"].isin(["Major", "Minor"])].copy()

In [97]:
mode_counts = (
    major_minor_data
    .groupby(["playlist_genre", "mode"])
    .size()
    .unstack(fill_value=0)
)

display(mode_counts)

mode,Major,Minor
playlist_genre,,
ambient,199,137
electronic,269,272
hip-hop,183,165
latin,185,213
pop,279,152


In [98]:
mode_summary = (
    major_minor_data
    .groupby(["playlist_genre", "mode"])["track_popularity"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
display(mode_summary)

,playlist_genre,mode,mean,std,count
0,ambient,Major,49.804020,16.165395,199
1,ambient,Minor,48.510949,15.700841,137
2,electronic,Major,48.773234,18.504252,269
3,electronic,Minor,53.558824,16.300121,272
4,hip-hop,Major,62.169399,16.975054,183
5,hip-hop,Minor,61.696970,19.020664,165
6,latin,Major,53.686486,21.724788,185
7,latin,Minor,56.122066,20.835251,213
8,pop,Major,62.397849,25.065442,279
9,pop,Minor,60.256579,26.628794,152


Performing a two-sided Welch's t-test to compare the mean popularity of Major and Minor songs using a significance level of $\alpha = 0.05$.
\begin{align}
H_o: \mu_{Major} = \mu_{minor}\\
H_1: \mu_{Major} \ne \mu_{minor}\\
\end{align}

In [109]:
from scipy.stats import ttest_ind

def welch_ttest(data, genre):
    major = data[
        (data["playlist_genre"] == genre) &
        (data["mode"] == "Major")
    ]["track_popularity"]

    minor = data[
        (data["playlist_genre"] == genre) &
        (data["mode"] == "Minor")
    ]["track_popularity"]

    t_stat, p_value = ttest_ind(major, minor, equal_var=False)
    return {
        "genre": genre,
        "t_stat": t_stat,
        "p_value": p_value,
        "major_mean": major.mean(),
        "minor_mean": minor.mean(),
        "difference in means": major.mean() - minor.mean(),
        "t_stat": t_stat,
        
    }

In [110]:
genres = major_minor_data["playlist_genre"].unique()
welch_results = pd.DataFrame([welch_ttest(major_minor_data, genre) for genre in genres])
welch_results["significant"] = welch_results["p_value"] < 0.05 
display(welch_results)

,genre,t_stat,p_value,major_mean,minor_mean,difference in means,significant
0,pop,0.814167,0.416207,62.397849,60.256579,2.141271,False
1,latin,-1.136926,0.256280,53.686486,56.122066,-2.435579,False
2,hip-hop,0.243402,0.807845,62.169399,61.696970,0.472429,False
3,ambient,0.732932,0.464176,49.804020,48.510949,1.293071,False
4,electronic,-3.190602,0.001504,48.773234,53.558824,-4.785589,True


Now we will perform a $95\%$ bootstrap confidence interval for each genre to see estimate the uncertainty around each difference in means.

In [115]:
def bootstrap_difference(data, genre, n_iterations = 10000):
    major = data[
        (data["playlist_genre"] == genre) &
        (data["mode"] == "Major")
    ]["track_popularity"]

    minor = data[
        (data["playlist_genre"] == genre) &
        (data["mode"] == "Minor")
    ]["track_popularity"]

    differences = []
    for _ in range(n_iterations):
        major_sample = np.random.choice(major, size=len(major), replace=True)
        minor_sample = np.random.choice(minor, size=len(minor), replace=True)
        differences.append(major_sample.mean() - minor_sample.mean())

    lower_bound = np.percentile(differences, 2.5)
    upper_bound = np.percentile(differences, 97.5)

    return {
        "genre": genre,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    }

In [118]:
boostrap_results = []
for genre in genres:
    result = bootstrap_difference(major_minor_data, genre)
    boostrap_results.append(result)

boostrap_results = pd.DataFrame(boostrap_results)
boostrap_results["significant"] = (boostrap_results["lower_bound"] > 0) | (boostrap_results["upper_bound"] < 0)
display(boostrap_results)

,genre,lower_bound,upper_bound,significant
0,pop,-2.996919,7.316327,False
1,latin,-6.654996,1.763862,False
2,hip-hop,-3.296771,4.285082,False
3,ambient,-2.165919,4.722230,False
4,electronic,-7.681789,-1.815828,True
